In [883]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, STL10
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import shapiro, normaltest
import importlib
import pandas as pd
import skimage.io as io
from math import log, sqrt
from mhnlib.fixed_points import get_symmetric_stability_matrix_gram, get_entropies, get_jacobian_gram, get_symmetric_stability_matrices_gram
from mhnlib.dynamics import DualDeterministicDynamics, StochasticDynamics

In [453]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Helper functions

In [ ]:
class PokemonDataset(Dataset):
    def __init__(
        self,
        root = "",
        variant="normal",
        game="red-blue",
        image_col="local_path",   # "local_path" = color, "bw_path" = black/white
        size=32,
        one_per_pokemon=True,
        mode="rgb",               # "rgb", "rgba", or "mask"
    ):
        self.root = root
        info = pd.read_csv(self.root + "pokemondb_sprites/metadata.csv")

        sub_info = info[
            (info["game"] == game)
            & (info["variant"] == variant)
        ].copy()

        sub_info = sub_info[
            sub_info[image_col].notna()
            & (sub_info[image_col].astype(str) != "")
        ]

        if one_per_pokemon:
            sub_info = sub_info.groupby("pokemon", as_index=False).first()

        self.paths = sub_info[image_col].astype(str).tolist()
        self.names = sub_info["pokemon"].astype(str).tolist()

        self.classes = sorted(set(self.names))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.labels = [self.class_to_idx[n] for n in self.names]

        self.size = size
        self.mode = mode

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.root + self.paths[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        img = Image.open(path).convert("RGBA")

        if self.size is not None:
            img = img.resize((self.size, self.size), Image.Resampling.NEAREST)

        arr = np.asarray(img).astype(np.float32) / 255.0

        if self.mode == "mask":
            # Binary alpha mask: [1, H, W]
            alpha = arr[..., 3]
            x = torch.from_numpy((alpha > 0).astype(np.float32)).unsqueeze(0)

        elif self.mode == "rgba":
            # Keep transparency: [4, H, W]
            x = torch.from_numpy(arr).permute(2, 0, 1)

        elif self.mode == "rgb":
            # Composite transparent background onto white: [3, H, W]
            rgb = arr[..., :3]
            alpha = arr[..., 3:4]
            bg = np.ones_like(rgb)
            rgb = alpha * rgb + (1.0 - alpha) * bg
            x = torch.from_numpy(rgb).permute(2, 0, 1)

        else:
            raise ValueError(f"Unknown mode: {self.mode}")

        return x, label

In [222]:
def shift_and_rms(x, eps=1e-12):
    shift = x.mean(dim=0, keepdim=True)
    x_shifted = x - shift

    rms = x_shifted.norm(dim=1).square().mean().sqrt()
    rms = rms.clamp_min(eps)

    x_norm = x_shifted / rms

    return shift, rms, x_norm

In [ ]:
@torch.no_grad()
def decode_latents(z, z_mean, z_std, vae_model, vae_scaling, batch_size=0, verbose=False):
    orig_ndim = z.ndim

    if z.ndim == 4:
        B, C, H, W = z.shape
        vae_input = z * z_std + z_mean

    elif z.ndim == 5:
        B1, B2, C, H, W = z.shape
        vae_input = z.reshape(B1 * B2, C, H, W)
        vae_input = vae_input * z_std + z_mean

    else:
        raise ValueError("z must have 4 or 5 dimensions.")

    vae_input = vae_input / vae_scaling

    if batch_size > 0:
        vae_decoded = []

        chunks = torch.split(vae_input, batch_size, dim=0)

        for chunk in tqdm(chunks, disable=not verbose):
            chunk = chunk.to(vae_model.device)
            decoded = vae_model.decode(chunk).sample.cpu()
            vae_decoded.append(decoded)

        vae_decoded = torch.cat(vae_decoded, dim=0)

    else:
        vae_decoded = vae_model.decode(
            vae_input.to(vae_model.device)
        ).sample.cpu()

    if orig_ndim == 5:
        vae_decoded = vae_decoded.view(B1, B2, *vae_decoded.shape[1:])

    return (1 + vae_decoded.clamp(-1, 1)) / 2

In [396]:
def play_images(
    images,
    time_variable,
    time_variable_name,
    num_cols,
    use_grayscale,
    base_image_size=5,
    pause=0.1,
):
    import time as time_module
    import numpy as np
    import matplotlib.pyplot as plt
    from IPython.display import display
    from skimage.color import rgb2gray

    num_runs = images.shape[0]
    num_times = images.shape[1]

    if num_cols is None:
        num_cols = num_runs

    num_rows = (num_runs + num_cols - 1) // num_cols

    fig, axs = plt.subplots(
        nrows=num_rows,
        ncols=num_cols,
        figsize=(base_image_size * num_cols, base_image_size * num_rows),
        squeeze=False,
    )

    axs = axs.flatten()
    ims = []

    def get_image(run, time_idx):
        img = images[run, time_idx]

        if hasattr(img, "detach"):
            img = img.detach().cpu()

        if img.shape[0] in (1, 3, 4):  # C, H, W
            img = img.permute(1, 2, 0)

        img = np.asarray(img)

        if img.shape[-1] == 1:
            img = img[..., 0]

        img = np.clip(img, 0, 1)

        if use_grayscale and img.ndim == 3:
            img = rgb2gray(img[..., :3])

        return img

    for run in range(num_runs):
        img = get_image(run, 0)

        if use_grayscale:
            im = axs[run].imshow(img, cmap="gray", vmin=0, vmax=1)
        else:
            im = axs[run].imshow(img)

        axs[run].axis("off")
        axs[run].set_title(f"run {run}")
        ims.append(im)

    for ax in axs[num_runs:]:
        ax.axis("off")

    fig.suptitle(
        f"Time step: {1}/{num_times}. "
        f"{time_variable_name}=${float(time_variable[0]):.2f}$",
        fontsize=20,
    )

    display_handle = display(fig, display_id=True)

    for time_idx in range(num_times):
        for run in range(num_runs):
            ims[run].set_data(get_image(run, time_idx))

        fig.suptitle(
            f"Time step: {time_idx + 1}/{num_times}. "
            f"{time_variable_name}=${float(time_variable[time_idx]):.2f}$",
            fontsize=20,
        )

        display_handle.update(fig)
        time_module.sleep(pause)

    plt.close(fig)


def animate_func(
    x,
    y,
    x0_vline,
    interval=100,
    repeat=True,
    fig_width=6,
    fig_height=4,
    save_path=None,
    x0_label=r"$x_0$",
    xscale="linear",   # "linear" or "log"
):
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.animation as animation

    x = np.asarray(x)
    y = np.asarray(y)

    if x.ndim != 1 or y.ndim != 1:
        raise ValueError(
            f"x and y must be 1D arrays. Got x.shape={x.shape}, y.shape={y.shape}"
        )

    if x.shape[0] != y.shape[0]:
        raise ValueError(
            f"x and y must have same length. Got {x.shape[0]} and {y.shape[0]}"
        )

    if xscale not in ["linear", "log"]:
        raise ValueError(f"xscale must be 'linear' or 'log'. Got {xscale}")
    if x0_vline is not None:
        if xscale == "log" and np.any(x <= 0):
            raise ValueError("For xscale='log', all x values must be positive.")

        if xscale == "log" and x0_vline <= 0:
            raise ValueError("For xscale='log', x0_vline must be positive.")

    num_times = len(x)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    line, = ax.plot([], [], lw=2)

    ax.set_xscale(xscale)
    if x0_vline is not None:
        ax.axvline(
            x0_vline,
            ls="dashed",
            color="black",
            label=x0_label,
        )

        ax.text(
            x0_vline,
            1.02,
            x0_label,
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
        )

    xmin, xmax = np.nanmin(x), np.nanmax(x)
    ymin, ymax = np.nanmin(y), np.nanmax(y)

    if xscale == "log":
        ax.set_xlim(xmin / 1.05, xmax * 1.05)
    else:
        dx = 0.05 * (xmax - xmin + 1e-12)
        ax.set_xlim(xmin - dx, xmax + dx)

    dy = 0.05 * (ymax - ymin + 1e-12)
    ax.set_ylim(ymin - dy, ymax + dy)

    title = ax.set_title("")
    if x0_vline is not None:
        ax.legend()

    def update(t):
        line.set_data(x[: t + 1], y[: t + 1])
        title.set_text(f"frame = {t}")
        return line, title

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=num_times,
        interval=interval,
        blit=False,
        repeat=repeat,
    )

    if save_path is not None:
        fps = 1000 / interval
        if save_path.endswith(".gif"):
            anim.save(save_path, writer=animation.PillowWriter(fps=fps))
        else:
            anim.save(save_path, writer="ffmpeg", fps=fps)
    plt.close(fig)
    return anim
    
def animate_images(
    images,
    time_variable,
    time_variable_name,
    num_cols,
    use_grayscale,
    base_image_size=5,
    interval=100,
    repeat=True,
    save_path=None,
):
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.animation as animation
    from skimage.color import rgb2gray

    num_runs = images.shape[0]
    num_times = images.shape[1]
    num_rows = (num_runs + num_cols - 1) // num_cols

    fig, axs = plt.subplots(
        nrows=num_rows,
        ncols=num_cols,
        figsize=(base_image_size * num_cols, base_image_size * num_rows),
        squeeze=False,
    )

    axs = axs.flatten()
    ims = []

    # leave room for the suptitle
    fig.subplots_adjust(top=0.88)

    def get_image(run, time_idx):
        img = images[run, time_idx]

        if hasattr(img, "detach"):
            img = img.detach().cpu()

        if img.shape[0] in (1, 3, 4):  # C, H, W
            img = img.permute(1, 2, 0)

        img = np.asarray(img)

        if img.ndim == 3 and img.shape[-1] == 1:
            img = img[..., 0]

        img = np.clip(img, 0, 1)

        if use_grayscale and img.ndim == 3:
            img = rgb2gray(img[..., :3])

        return img

    for run in range(num_runs):
        img = get_image(run, 0)

        if use_grayscale:
            im = axs[run].imshow(img, cmap="gray", vmin=0, vmax=1)
        else:
            im = axs[run].imshow(img)

        axs[run].axis("off")
        axs[run].set_title(f"run {run}")
        ims.append(im)

    for ax in axs[num_runs:]:
        ax.axis("off")

    title = fig.suptitle(
        f"{time_variable_name}: {float(time_variable[0]):.4f} | frame: 0/{num_times-1}",
        fontsize=20,
    )

    def update(time_idx):
        for run in range(num_runs):
            ims[run].set_data(get_image(run, time_idx))

        t = float(time_variable[time_idx])
        title.set_text(
            f"{time_variable_name}: {t:.4f} | frame: {time_idx}/{num_times-1}"
        )

        return ims + [title]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=num_times,
        interval=interval,
        blit=False,   # important for suptitle visibility
        repeat=repeat,
    )

    if save_path is not None:
        if save_path.endswith(".gif"):
            anim.save(save_path, writer="pillow", dpi=120)
        else:
            anim.save(save_path, writer="ffmpeg", dpi=120)

    plt.close(fig)
    return anim

### Dataset selection

In [5]:
torch.manual_seed(1101252)
IMG_SIZE = 32
DATASET = "cifar100"  # "cifar100", "stl10", "mnist", or "pokemon"

tfm = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
])

if DATASET == "cifar100":
    dataset = CIFAR100(root="datasets/cifar100/", train=True, download=True, transform=tfm)
    test_set = CIFAR100(root="datasets/cifar100/", train=False, download=True, transform=tfm)

elif DATASET == "stl10":
    dataset = STL10(root="datasets/stl10/", split="train", download=True, transform=tfm)
    test_set = STL10(root="datasets/stl10/", split="test", download=True, transform=tfm)

elif DATASET == "mnist":
    dataset = MNIST(root="datasets/mnist/", train=True, download=True, transform=tfm)
    test_set = MNIST(root="datasets/mnist/", train=False, download=True, transform=tfm)
elif DATASET == "pokemon":
    dataset = PokemonDataset(root="datasets/", variant="normal", game="red-blue", image_col="local_path", size=IMG_SIZE, one_per_pokemon=True, mode="rgb")
    test_set = None
num_classes = len(dataset.classes)
print(f"Number of classes: {num_classes}")

Number of classes: 100


### Load Autoencoder

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [6]:
from diffusers import AutoencoderKL
ae_model = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
ae_model = ae_model.to(device).eval()
ae_model.requires_grad_(False)
ae_scaling = ae_model.config.scaling_factor

In [7]:
torch.random.manual_seed(1101252)
MHN_BATCH_SIZE = 1024
mhn_data_loader = DataLoader(
    dataset,
    batch_size=MHN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)
data = []
latents = []
latents_labels = []
for (batch_x, batch_y) in tqdm(mhn_data_loader, desc="Encoding images with VAE"):
    batch_x = batch_x.to(device)
    with torch.no_grad():
        batch_x = batch_x.expand(-1, 3, -1, -1) if batch_x.shape[1] == 1 else batch_x
        z = ae_model.encode(batch_x).latent_dist.mode() * ae_scaling
    data.append(batch_x.cpu())
    latents.append(z.cpu())
    latents_labels.append(batch_y)
data = torch.cat(data, dim=0)
latents = torch.cat(latents, dim=0)
latents_labels = torch.cat(latents_labels, dim=0)

Encoding images with VAE:   0%|          | 0/49 [00:00<?, ?it/s]

In [8]:
C_latents, H_latents, W_latents = latents.shape[1:]

### Subsample latents

In [454]:
# sample some latents according to label
torch.random.manual_seed(1101252)
samples_per_class = 1
sampled_latents = []
sampled_labels = []
for c in range(num_classes):
    class_indices = (latents_labels == c).nonzero(as_tuple=True)[0]
    sampled_indices = torch.randperm(len(class_indices))[:samples_per_class]
    sampled_latents.append(latents[class_indices[sampled_indices]])
    sampled_labels.append(latents_labels[class_indices[sampled_indices]])
sampled_latents = torch.cat(sampled_latents, dim=0)
sampled_labels = torch.cat(sampled_labels, dim=0)

In [455]:
patterns = sampled_latents.reshape(sampled_latents.shape[0],-1)
patterns_shift, patterns_rms, patterns_centered_scaled = shift_and_rms(patterns)

### Quenched $w$ fixed points

In [321]:
%autoreload 2

In [330]:
dds_cs = DualDeterministicDynamics(patterns_centered_scaled)
gram_cs = dds_cs.gram
dds_cs = dds_cs.to(device)
uniform_weights = torch.ones(gram_cs.shape[0])/gram_cs.shape[0]
stability_matrix_patterns_cs = get_symmetric_stability_matrix_gram(gram_cs,uniform_weights )

In [331]:
torch.random.manual_seed(1101252)
beta_c = 1.0/torch.linalg.eigvalsh(stability_matrix_patterns_cs)[-1]
log10_beta_c = torch.log10(beta_c)
betas = torch.logspace(log10_beta_c - 0.5, log10_beta_c + 2.0, 101) 

In [332]:
num_ics = 60
perturbed_initial_conditions = torch.distributions.Dirichlet(100*torch.ones(gram_cs.shape[0])).sample((num_ics,))
quenched_weights = dds_cs.integrate(
                    perturbed_initial_conditions.to(device),
                    betas.to(device),
                    num_iterations=5000,
                    verbose=True
                ).cpu()

  0%|          | 0/5000 [00:00<?, ?it/s]

In [333]:
quenched_fp_patterns = torch.einsum('rbk,ki->rbi', quenched_weights, patterns_centered_scaled)
latents_fp_patterns = quenched_fp_patterns.view((quenched_fp_patterns.shape[0], quenched_fp_patterns.shape[1], C_latents, H_latents, W_latents))
latents_fp_shift = patterns_center.view(C_latents, H_latents, W_latents)
quenched_fp_images = decode_latents(latents_fp_patterns, latents_fp_shift, patterns_rms, ae_model, ae_scaling, batch_size=64, verbose=True)

  0%|          | 0/95 [00:00<?, ?it/s]

In [334]:
_ = animate_images(quenched_fp_images, betas, "$\\beta$", num_cols=10, use_grayscale=False, base_image_size=5, interval=200, repeat=False, save_path=f"animations/mhn_quenched_fixed_points_samples_per_classes={samples_per_class}.mp4")

In [335]:
_ = animate_func(betas.numpy(), get_entropies(quenched_weights).mean(dim=0) , x0_vline=beta_c, x0_label="$\\beta_c$", interval=200, xscale="log", save_path=f"animations/mhn_quenched_fixed_points_entropies_samples_per_classes={samples_per_class}.mp4")

### Annealing

In [401]:
sdn_cs = StochasticDynamics(patterns_centered_scaled, biases=torch.zeros(patterns_centered_scaled.shape[0]))
sdn_cs = sdn_cs.to(device)
gram_cs = patterns_centered_scaled @ patterns_centered_scaled.T
uniform_weights = torch.ones(gram_cs.shape[0])/gram_cs.shape[0]
stability_matrix_patterns_cs = get_symmetric_stability_matrix_gram(gram_cs,uniform_weights )

In [408]:
torch.random.manual_seed(1101252)
beta_c = 1.0/torch.linalg.eigvalsh(stability_matrix_patterns_cs)[-1]
log10_beta_c = torch.log10(beta_c)
betas_annealing = torch.logspace(log10_beta_c - 3, log10_beta_c + 3.5, 101)

In [447]:
torch.random.manual_seed(1101252)
num_ics = 100
initial_conditions_weights = torch.distributions.Dirichlet(torch.ones(patterns_centered_scaled.shape[0])).sample((num_ics,))
initial_conditions = initial_conditions_weights @ patterns_centered_scaled
annealed_patterns = []
for beta_idx in tqdm(range(len(betas_annealing))):
    if len(annealed_patterns) == 0:
        p_ic = initial_conditions[:,None,:]
    else:
        p_ic = annealed_patterns[-1]
    retrieved_patterns = sdn_cs.integrate(p_ic.to(device), betas_annealing[beta_idx:beta_idx+1].to(device), dt=0.01, num_iterations=3000, verbose=False)
    annealed_patterns.append(retrieved_patterns.cpu())
annealed_patterns = torch.cat(annealed_patterns, dim=1)

  0%|          | 0/101 [00:00<?, ?it/s]

In [448]:
latents_annealed_patterns = annealed_patterns.view((annealed_patterns.shape[0], annealed_patterns.shape[1], C_latents, H_latents, W_latents))
latents_annealed_patterns_shift = patterns_center.view(C_latents, H_latents, W_latents)
annealed_images = decode_latents(latents_annealed_patterns, latents_annealed_patterns_shift, patterns_rms, ae_model, ae_scaling, batch_size=64, verbose=True)

  0%|          | 0/158 [00:00<?, ?it/s]

In [449]:
_ = animate_images(annealed_images, betas_annealing, "$\\beta$", num_cols=10, use_grayscale=False, base_image_size=5, interval=200, repeat=False, save_path=f"animations/mhn_annealed_fixed_points_samples_per_classes={samples_per_class}.mp4")

In [450]:
similarities = torch.einsum('ki,sbi->sbk',patterns_centered_scaled, annealed_patterns )
annealed_patterns_norms = annealed_patterns.norm(dim=-1)
patterns_centered_scaled_norms = patterns_centered_scaled.norm(dim=-1)
similarities = similarities/(patterns_centered_scaled_norms[None,None,:]*annealed_patterns_norms[:, :, None])

In [451]:
_ = animate_func(betas_annealing.numpy(), similarities.max(dim=-1).values.mean(dim=0) , x0_vline=None, x0_label=None, interval=200, xscale="log", save_path=f"animations/mhn_annealed_fixed_points_max_cos_samples_per_classes={samples_per_class}.mp4")

### All fixed points

In [840]:
def generate_bifurcation_ic(gram, w0, eig_tol, num_trials, perturbation_std, eps = 1e-12, return_info=True):
    stab_m, proj_m = get_symmetric_stability_matrix_gram(gram,w0, return_proj=True)
    evals, evecs = torch.linalg.eigh(stab_m)
    lambda_max = evals[-1]
    gap_mask = evals >= lambda_max - eig_tol * torch.abs(lambda_max).clamp_min(eps)
    U = evecs[:, gap_mask]       # shape: (K, m)
    m = U.shape[1]
    # Random combinations inside the nearly-degenerate leading eigenspace
    coeffs = torch.randn(
        num_trials, m,
        device=gram.device,
        dtype=gram.dtype,
    )
    u = coeffs @ U.T             # shape: (num_trials, K)
    _, C_invsqrt = fisher_sqrt_and_invsqrt(w0, eps=eps)

    delta_h = u @ C_invsqrt.T
    delta_h = delta_h - delta_h.mean(dim=-1, keepdim=True)
    delta_h = delta_h / delta_h.std(dim=-1, keepdim=True).clamp_min(eps)
    h0 = torch.log(w0.clamp_min(eps))
    h = h0[None, :] + perturbation_std * delta_h

    w_ic = torch.softmax(h, dim=-1)
    if return_info:
        info = {
            "evals": evals,
            "lambda_max": lambda_max,
            "num_unstable_modes": m,
            "leading_evecs_sym": U,
            "delta_h": delta_h,
            "stab_m": stab_m,
            "proj_m": proj_m,
        }
        return w_ic, info
    return w_ic

In [882]:
K = patterns_centered_scaled.shape[0]
dd_dyn = DualDeterministicDynamics(patterns_centered_scaled)
gram_cs = dd_dyn.gram
dd_dyn = dd_dyn.to(device)

In [887]:
w0 = torch.ones(K)/K
stab_0, proj_0, info_0 = get_symmetric_stability_matrices_gram(gram, w0, return_fisher=True, return_proj=True)
vals_stab_0, vecs_stab_0 = torch.linalg.eigh(gram)


In [876]:

num_trials=10000
w_ics,  infos  = generate_bifurcation_ic(gram_cs, w0, eig_tol=0.1, num_trials=num_trials,  perturbation_std=1.0, )
delta_beta = 0.001
beta = 1.0/infos['lambda_max']
max_num_beta_trials = 11
for beta_step in range(max_num_beta_trials):
    trial_w_fps = dd_dyn.integrate(w_ics.to(device), (fp_beta_c + (beta_step+1)*delta_beta).view(1).to(device) , num_iterations=5000, verbose=True).cpu()[:,0,:]
    new_trial_beta_cs = torch.as_tensor(new_trial_beta_cs)
    new_trial_mask = ((new_trial_beta_cs-fp_beta_c).abs() > delta_beta)*(new_trial_beta_cs > fp_beta_c)
    if new_trial_mask.sum() == 0:
        continue
    new_trial_candidate_beta_cs = new_trial_beta_cs[new_trial_mask]
    new_trial_candidate_fps = new_trial_fps[new_trial_mask]

  0%|          | 0/5000 [00:00<?, ?it/s]

In [805]:
import networkx as nx
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
tolerance = 4/K
delta_beta = 0.0001
max_num_beta_trials = 11
for beta_step in range(max_num_beta_trials):
    new_trial_fps = dd_dyn.integrate(fp_w_perturbations.to(device), (fp_beta_c + (beta_step+1)*delta_beta).view(1).to(device) , num_iterations=5000, verbose=True).cpu()[:,0,:]
    new_trial_beta_cs  = []
    for new_trial_fp in new_trial_fps:
        new_trial_fp_beta_c, new_trial_fp_u_vec, new_trial_fp_proj =  fixed_point_neighborhood(gram_cs,new_trial_fp)
        new_trial_beta_cs.append(new_trial_fp_beta_c)
    new_trial_beta_cs = torch.as_tensor(new_trial_beta_cs)
    new_trial_mask = ((new_trial_beta_cs-fp_beta_c).abs() > delta_beta)*(new_trial_beta_cs > fp_beta_c)
    if new_trial_mask.sum() == 0:
        continue
    new_trial_candidate_beta_cs = new_trial_beta_cs[new_trial_mask]
    new_trial_candidate_fps = new_trial_fps[new_trial_mask]

    #num_new_components = PCA(n_components='mle').fit(new_trial_candidate_fps).n_components_

    #clustering = KMeans(n_clusters=num_new_components).fit(new_trial_candidate_fps)
    
    candidates_overlap = new_trial_candidate_fps @ new_trial_candidate_fps.T
    candidates_sim = candidates_overlap > tolerance
    candidates_sim[torch.arange(0,candidates_sim.shape[0]), torch.arange(0,candidates_sim.shape[0])] = True
    
    candidates_sim_graph = nx.from_numpy_array(candidates_sim.numpy())
    candidates_cc = list(nx.connected_components(candidates_sim_graph))
    final_candidates_indices = torch.as_tensor([ list(ccc)[0] for ccc in candidates_cc])
    new_fps = new_trial_candidate_fps[final_candidates_indices]
    new_beta_cs = new_trial_candidate_beta_cs[final_candidates_indices]
    break

  0%|          | 0/5000 [00:00<?, ?it/s]

In [806]:
candidates_overlap

tensor([[0.0100, 0.0100, 0.0100,  ..., 0.0099, 0.0099, 0.0099],
        [0.0100, 0.0100, 0.0100,  ..., 0.0099, 0.0099, 0.0099],
        [0.0100, 0.0100, 0.0100,  ..., 0.0099, 0.0099, 0.0099],
        ...,
        [0.0099, 0.0099, 0.0099,  ..., 0.9964, 0.9964, 0.9964],
        [0.0099, 0.0099, 0.0099,  ..., 0.9964, 0.9964, 0.9964],
        [0.0099, 0.0099, 0.0099,  ..., 0.9964, 0.9964, 0.9964]])